# Comparison — BERT vs DistilBERT Across All Tasks

**Run after:** all three pipeline notebooks have saved their results to `results/`

## What this notebook does
Loads saved results from all pipelines and produces:
- Final accuracy comparison table (our results vs paper)
- Model size and speed comparison
- Plots for the presentation

In [ ]:
!pip install -q matplotlib seaborn pandas

In [ ]:
import sys
sys.path.append('../src')

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_results

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
# ── Load all results ───────────────────────────────────────────────────────
RESULTS_DIR = '../results'

def safe_load(filename):
    try:
        return load_results(filename)
    except:
        print(f'Warning: {filename} not found — run the pipeline notebook first')
        return None

bert_sst2      = safe_load('sst2_bert_base_uncased.json')
distil_sst2    = safe_load('sst2_distilbert_base_uncased.json')
bert_imdb      = safe_load('imdb_bert_base_uncased.json')
distil_imdb    = safe_load('imdb_distilbert_base_uncased.json')
bert_squad     = safe_load('squad_bert_base_uncased.json')
distil_squad   = safe_load('squad_distilbert_base_uncased.json')

In [ ]:
# ── Accuracy comparison table ──────────────────────────────────────────────
def get_acc(result):
    return result['val_accuracy'] if result else 'N/A'

rows = [
    {
        'Model':   'BERT-base (paper)',
        'SST-2':   93.5,
        'IMDb':    95.63,
        'SQuAD F1': 88.5,
    },
    {
        'Model':   'DistilBERT (paper)',
        'SST-2':   91.3,
        'IMDb':    95.79,
        'SQuAD F1': 85.8,
    },
    {
        'Model':   'BERT-base (ours)',
        'SST-2':   get_acc(bert_sst2),
        'IMDb':    get_acc(bert_imdb),
        'SQuAD F1': 'see pipeline_squad',
    },
    {
        'Model':   'DistilBERT (ours)',
        'SST-2':   get_acc(distil_sst2),
        'IMDb':    get_acc(distil_imdb),
        'SQuAD F1': 'see pipeline_squad',
    },
]

df = pd.DataFrame(rows)
print('=== Accuracy Comparison ===')
print(df.to_string(index=False))

In [ ]:
# ── Model size and speed table ─────────────────────────────────────────────
if bert_sst2 and distil_sst2:
    size_rows = [
        {
            'Model':            'BERT-base',
            'Parameters':       f"{bert_sst2['total_parameters']:,}",
            'Size (MB)':        bert_sst2['model_size_mb'],
            'Speed (samples/s)': bert_sst2['inference_speed']['samples_per_second'],
        },
        {
            'Model':            'DistilBERT',
            'Parameters':       f"{distil_sst2['total_parameters']:,}",
            'Size (MB)':        distil_sst2['model_size_mb'],
            'Speed (samples/s)': distil_sst2['inference_speed']['samples_per_second'],
        },
    ]
    df_size = pd.DataFrame(size_rows)
    print('\n=== Model Size & Speed ===')
    print(df_size.to_string(index=False))

    speedup = round(distil_sst2['inference_speed']['samples_per_second'] /
                    bert_sst2['inference_speed']['samples_per_second'], 2)
    param_reduction = round((1 - distil_sst2['total_parameters'] /
                              bert_sst2['total_parameters']) * 100, 1)
    print(f'\nDistilBERT is {speedup}x faster and {param_reduction}% smaller than BERT-base')

In [ ]:
# ── Plot: accuracy across tasks ────────────────────────────────────────────
if bert_sst2 and distil_sst2 and bert_imdb and distil_imdb:
    tasks  = ['SST-2', 'IMDb']
    bert_scores   = [bert_sst2['val_accuracy'],   bert_imdb['val_accuracy']]
    distil_scores = [distil_sst2['val_accuracy'], distil_imdb['val_accuracy']]

    x = range(len(tasks))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar([i - width/2 for i in x], bert_scores,   width, label='BERT-base',  color='#4C72B0')
    bars2 = ax.bar([i + width/2 for i in x], distil_scores, width, label='DistilBERT', color='#DD8452')

    ax.set_ylabel('Accuracy (%)')
    ax.set_title('BERT-base vs DistilBERT — Our Results')
    ax.set_xticks(list(x))
    ax.set_xticklabels(tasks)
    ax.set_ylim(85, 100)
    ax.legend()

    for bar in bars1 + bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('../results/accuracy_comparison.png', dpi=150)
    plt.show()
    print('Plot saved to results/accuracy_comparison.png')